# 08 · FlashAttention from scratch: online softmax and tiling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/architecture/08_flash_attention.ipynb) &nbsp; 📖 Concept deck: [FlashAttention internals](https://nilesh-salpe.github.io/visual-ai/architecture/09-flash-attention/) &nbsp; ⏱️ Suggested time: **50 min**

**You will build:** the online-softmax recurrence, a tiled FlashAttention-2-style forward pass that never materializes the N×N matrix, and an HBM-traffic model that shows why it is faster.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _naive_attention(Q, K, V, causal=False):
    S = Q @ K.T / np.sqrt(Q.shape[1])
    if causal:
        S = np.where(np.tril(np.ones(S.shape, bool)), S, -np.inf)
    return softmax(S) @ V

## Task 1: Online softmax (Milakov & Gimelshein, 2018)

Softmax normally needs the max of the whole row before exponentiating. The online version streams through the row **in chunks**, keeping a running max $m$ and a running sum $\ell$:
$$m_{new} = \max(m, \max(\text{chunk})),\qquad \ell \leftarrow \ell\,e^{m-m_{new}} + \textstyle\sum e^{\text{chunk}-m_{new}}$$
When the max goes up, the old sum is **rescaled** by $e^{m-m_{new}}$. After the last chunk, softmax $= e^{x - m}/\ell$.
Return `(m, l)` computed with the given chunk size. (The test then builds the softmax from them.)

In [ ]:
def online_softmax(x, chunk):
    """x: 1-D array. Returns (m, l): final running max and running sum of exp(x - m)."""
    m, l = -np.inf, 0.0
    # TODO: loop over x[i:i+chunk], update m and l
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_online_softmax(f):
    r = np.random.default_rng(1); x = r.normal(0, 5, 37)
    for chunk in [1, 4, 10, 37]:
        m, l = f(x, chunk)
        assert np.isclose(m, x.max()), f"chunk={chunk}: running max wrong"
        assert np.allclose(np.exp(x - m) / l, softmax(x)), f"chunk={chunk}: exp(x - m) / l must equal softmax(x). Did you rescale the old sum?"

check("online_softmax", _test_online_softmax, online_softmax)

## Task 2: Tiled attention forward (FlashAttention-2 loop order)

Split $Q$ into row blocks of size $B_r$ and $K, V$ into column blocks of size $B_c$. For each $Q$ block (the **outer** loop, which FA2 runs in parallel across thread blocks), stream over K/V blocks (the **inner** loop) and keep, per row, a running max $m$, running sum $\ell$ and **unnormalized** output $\tilde O$:
```
S = Q_i K_j^T / sqrt(d)            # [Br, Bc] lives only in SRAM
m_new = max(m, rowmax(S))
P = exp(S - m_new)
l = l * exp(m - m_new) + rowsum(P)
O~ = O~ * exp(m - m_new) + P @ V_j  # rescale old partial output, add new
m = m_new
```
After the inner loop: $O_i = \tilde O / \ell$, and save the log-sum-exp $L_i = m + \log \ell$ for the backward pass (FA2 stores only this one number per row).
Support `causal=True`: mask entries with column > row, and skip K/V blocks that lie entirely in the future.

In [ ]:
def flash_attention(Q, K, V, Br=4, Bc=4, causal=False):
    """Q, K, V: [N, d]. Returns (O [N, d], L [N] log-sum-exp per row). Never build an [N, N] array."""
    N, d = Q.shape; O = np.zeros((N, d)); L = np.zeros(N)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_flash_attention(f):
    r = np.random.default_rng(2)
    for N, d, Br, Bc in [(16, 8, 4, 4), (19, 8, 5, 3), (7, 4, 16, 16), (33, 16, 8, 1)]:
        Q, K, V = r.normal(size=(3, N, d))
        for causal in [False, True]:
            O, L = f(Q, K, V, Br, Bc, causal)
            assert np.allclose(O, _naive_attention(Q, K, V, causal), atol=1e-10), f"N={N} Br={Br} Bc={Bc} causal={causal}: output differs from naive attention"
            S = Q @ K.T / np.sqrt(d)
            if causal: S = np.where(np.tril(np.ones((N, N), bool)), S, -np.inf)
            mx = S.max(1); assert np.allclose(L, mx + np.log(np.exp(S - mx[:, None]).sum(1))), "L must be the row-wise log-sum-exp"

check("flash_attention", _test_flash_attention, flash_attention)

## Task 3: Count the memory traffic

Model HBM reads and writes in **elements** for one head (forward pass only):
* **Standard** (Algorithm 0 in the FA1 paper): read Q, K and write S ($2Nd + N^2$); read S, write P ($2N^2$); read P, V and write O ($N^2 + 2Nd$).
* **Flash** (FA2 loop order): read each $Q_i$ once and write each $O_i$ once ($2Nd$); for each of the $\lceil N/B_r\rceil$ query blocks, read all of K and V ($2Nd$ each time).

Return a dict `{'standard': ..., 'flash': ...}`. The FA1 paper's Theorem 2 gives $\Theta(Nd + N^2)$ vs $\Theta(N^2d^2M^{-1})$, where block size $\propto M/d$ for SRAM size $M$.

In [ ]:
def hbm_traffic(N, d, Br):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_hbm_traffic(f):
    t = f(1024, 64, 64)
    assert t['standard'] == 4 * 1024 * 64 + 4 * 1024 ** 2, f"standard: got {t['standard']:,}"
    assert t['flash'] == 2 * 1024 * 64 + 16 * 2 * 1024 * 64, f"flash: got {t['flash']:,}"
    assert f(4096, 64, 128)['flash'] < f(4096, 64, 128)['standard'] / 3, "with d=64 and Br=128, flash should move under a third of the data"
    assert f(4096, 64, 256)['flash'] < f(4096, 64, 128)['flash'], "bigger query tiles mean fewer K/V re-reads"

check("hbm_traffic", _test_hbm_traffic, hbm_traffic)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Online softmax (Milakov & Gimelshein, 2018)
def online_softmax_solution(x, chunk):
    m, l = -np.inf, 0.0
    for i in range(0, len(x), chunk):
        c = x[i:i + chunk]; m_new = max(m, c.max())
        l = l * np.exp(m - m_new) + np.exp(c - m_new).sum()
        m = m_new
    return m, l

check("online_softmax (reference)", _test_online_softmax, online_softmax_solution)

In [ ]:
# @title Solution for Task 2: Tiled attention forward (FlashAttention-2 loop order)
def flash_attention_solution(Q, K, V, Br=4, Bc=4, causal=False):
    N, d = Q.shape; O = np.zeros((N, d)); L = np.zeros(N); scale = 1 / np.sqrt(d)
    for i0 in range(0, N, Br):                      # outer loop: query blocks (parallel on GPU)
        Qi = Q[i0:i0 + Br]; rows = np.arange(i0, i0 + len(Qi))
        m = np.full(len(Qi), -np.inf); l = np.zeros(len(Qi)); acc = np.zeros((len(Qi), d))
        for j0 in range(0, N, Bc):                  # inner loop: stream K/V blocks through "SRAM"
            if causal and j0 > rows[-1]:
                break                                # whole block is in the future
            S = Qi @ K[j0:j0 + Bc].T * scale
            if causal:
                S = np.where(np.arange(j0, j0 + S.shape[1])[None, :] <= rows[:, None], S, -np.inf)
            m_new = np.maximum(m, S.max(1))
            P = np.exp(S - m_new[:, None])
            corr = np.exp(m - m_new)
            l = l * corr + P.sum(1)
            acc = acc * corr[:, None] + P @ V[j0:j0 + Bc]
            m = m_new
        O[i0:i0 + Br] = acc / l[:, None]
        L[i0:i0 + Br] = m + np.log(l)
    return O, L

check("flash_attention (reference)", _test_flash_attention, flash_attention_solution)

In [ ]:
# @title Solution for Task 3: Count the memory traffic
def hbm_traffic_solution(N, d, Br):
    standard = (2 * N * d + N * N) + (2 * N * N) + (N * N + 2 * N * d)
    flash = 2 * N * d + int(np.ceil(N / Br)) * 2 * N * d
    return {'standard': standard, 'flash': flash}

check("hbm_traffic (reference)", _test_hbm_traffic, hbm_traffic_solution)

---
## Experiments
**Traffic and memory as the sequence grows** (d = 64 as in GPT-2 medium, B_r = 128, BF16 = 2 bytes). In this simple forward-only model, standard attention moves about $4N^2$ elements (S and P written and read back), while Flash re-reads K and V once per query block, about $2N^2d/B_r$. With $B_r = 2d$ that is about 4× less traffic, and larger tiles help further, up to what SRAM can hold. The bigger win in this model is **memory**: Flash's extra state is $O(N)$ (the $m$, $\ell$ or $L$ per row) instead of an $O(N^2)$ matrix per head.

The paper measures much more: on GPT-2 medium (N = 1024, d = 64, forward + backward, A100), standard attention did **66.6 GFLOPs, 40.3 GB of HBM traffic, 41.7 ms**, and FlashAttention did **75.2 GFLOPs** (more, because of recomputation), **4.4 GB, 7.3 ms** (FA1 Fig. 2). Real gains include the backward pass (which no longer reads S and P) and fusing mask, softmax and dropout into one kernel instead of several separate passes over HBM.

In [ ]:
for N in [1024, 4096, 16384, 65536]:
    t = hbm_traffic_solution(N, 64, 128)
    print(f"N={N:6,}: standard {2 * t['standard'] / 2**20:9,.0f} MiB moved, flash {2 * t['flash'] / 2**20:7,.0f} MiB  ({t['standard'] / t['flash']:4.1f}x less);"
          f"  N×N score matrix alone = {2 * N * N / 2**20:8,.0f} MiB per head")
print("\nFA1 paper, GPT-2 medium fwd+bwd on A100: standard 40.3 GB / 41.7 ms vs FlashAttention 4.4 GB / 7.3 ms (9.2x less traffic, 5.7x faster)")
r = np.random.default_rng(0); Q, K, V = r.normal(size=(3, 512, 64))
O, _ = flash_attention_solution(Q, K, V, 64, 64, causal=True)
print(f"\nmax |flash - naive| at N=512, causal: {np.abs(O - _naive_attention(Q, K, V, True)).max():.1e}  (exact up to float rounding)")

---
## Takeaways
* The online-softmax rescaling trick lets attention be computed block by block, without ever holding a full row of scores.
* Tiling keeps S and P in on-chip SRAM. HBM sees only Q, K, V and O, so memory is linear in N, and the speedup comes from moving fewer bytes, not from fewer FLOPs.
* FA2's changes: loop over query blocks on the outside (parallel across thread blocks), rescale the output only once at the end, and store just the log-sum-exp for the backward pass, which recomputes S and P on chip instead of reading them.